# TW-PLD-LightGCL  Phase 1: LightGCL baseline on Gowalla

This notebook trains the **LightGCL backbone only** (no PLD, no temporal weighting yet) on the processed Gowalla benchmark to reproduce the published numbers.

**Target** (from the LightGCL paper, Cai et al. 2023, Table 3): Recall@20 ~= 0.18, NDCG@20 ~= 0.10 on Gowalla.

## One-time setup before running

1. **Runtime  Change runtime type  GPU** (T4 is fine; A100 if available).
2. **Generate a fine-grained Personal Access Token** at <https://github.com/settings/personal-access-tokens>:
   - Repository access: **Only select repositories  TW-PLD-LightGCL**
   - Permissions  Repository permissions  **Contents: Read**
   - Expiration: 90 days (or whatever fits your timeline)
3. **Add to Colab Secrets**: click the key icon in the left sidebar  add new secret named `GITHUB_PAT`, paste your token, toggle **Notebook access** ON.

If you skip step 3, the auth cell will prompt you to paste the token at runtime instead.

## Expected runtime

- GPU sanity run (1 epoch): ~1 min on T4
- Full 100-epoch training: ~30-60 min on T4, faster on A100


## 1. Verify GPU runtime

In [ ]:
!nvidia-smi -L
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU. Runtime  Change runtime type  GPU.")


## 2. Clone the repo (or pull latest if already cloned)

In [ ]:
import os, subprocess

REPO = "marisfemale/TW-PLD-LightGCL"
LOCAL_DIR = "/content/TW-PLD-LightGCL"

try:
    from google.colab import userdata
    token = userdata.get('GITHUB_PAT')
    print("Got PAT from Colab Secrets.")
except Exception:
    from getpass import getpass
    token = getpass(f'GitHub PAT with Contents:Read on {REPO}: ')

if not os.path.exists(LOCAL_DIR):
    subprocess.run(['git', 'clone', f'https://{token}@github.com/{REPO}.git', LOCAL_DIR],
                   check=True)
else:
    subprocess.run(['git', '-C', LOCAL_DIR, 'pull'], check=True)

%cd $LOCAL_DIR
!git log --oneline -3


## 3. Install dependencies

Colab ships with torch + CUDA pre-installed. Just ensure the rest are present.

In [ ]:
!pip install --quiet pandas scipy tqdm


## 4. GPU sanity run (1 epoch, full pipeline)

Confirms the model trains, evaluates, and saves checkpoints on this GPU. Don't proceed to full training unless this passes.

In [ ]:
!python src/train.py --device cuda \
    --epochs 1 --eval-every-epochs 1 \
    --run-name gpu_sanity


## 5. Full training (100 epochs, early stopping on val Recall@20)

This is the real run for Phase 1 baseline reproduction.

In [ ]:
!python src/train.py --device cuda \
    --epochs 100 --eval-every-epochs 3 \
    --early-stop-patience 10 \
    --run-name lightgcl_gowalla_baseline


## 6. Summary of the run

In [ ]:
import csv, json
from pathlib import Path

run_dir = Path('outputs/lightgcl_gowalla_baseline')

print('=== Config ===')
print(json.dumps(json.load((run_dir / 'config.json').open()), indent=2))

print('\n=== Metrics (per-eval + final test) ===')
with (run_dir / 'metrics.csv').open() as f:
    for r in csv.DictReader(f):
        cleaned = {k: v for k, v in r.items() if v}
        print(cleaned)


## 7. Zip and download outputs

Bundles `outputs/lightgcl_gowalla_baseline/` (config + metrics + best checkpoint + log) and downloads it to your machine. About 20 MB.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('lightgcl_gowalla_baseline_run', 'zip',
                              root_dir='outputs/lightgcl_gowalla_baseline')
print(f'Archive: {archive}')
files.download(archive)
